<h1>Chapter 7 - Evaluation</h1>
<i>Measuring whether your `TinyAgent` actually works.</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 7 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>

### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter.

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to **Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM - `Gemma 4`

We use the same LLM that we used previously, namely the Gemma 4 E4B model with native tool calling and reasoning capabilities. Note that it will also be used to judge itself, as we will explore in the LLM-as-a-judge section.

In [1]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B (with native thinking and tool calling)
llm = LLM(model="gemma4:e4b", think=True)

If you want to use another LLM, here are a couple of options that use the `OpenAI` library:

In [2]:
# from openai import OpenAI
# from illustrated_agents.llm import OpenAIClientLLM

# # Ollama through OpenAI API
# client = OpenAI(base_url="http://localhost:11434/v1/", api_key="no_key")
# llm = OpenAIClientLLM(model="gemma4:e4b", client=client, think=True)

# # Llama.cpp server
# client = OpenAI(base_url="http://localhost:8080/v1/", api_key="no_key")
# llm = OpenAIClientLLM(model="gemma-4-E4B-it-Q4_K_M", client=client, think=True)

# # LM Studio
# client = OpenAI(base_url="http://localhost:1234/v1/", api_key="no_key")
# llm = OpenAIClientLLM(model="gemma-4-e4b-it", client=client, think=True)

# # Google's Gemini / Gemma
# client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="YOUR_GEMINI_API_KEY")
# llm = OpenAIClientLLM(model="gemini-2.5-flash", client=client, think=True)

## 2 - Building the `Evaluator`

In [3]:
from dataclasses import dataclass
from typing import Callable

# Type hint for the scorers
##  (prediction: str, example: dict) -> bool | float
Scorer = Callable[[str, dict], bool | float]

@dataclass
class Benchmark:
    name: str
    examples: list[dict]
    scorer: Callable

In [4]:
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import NativeTools
from illustrated_agents.chapters.ch6 import NativeReAct, TinyAgent


class Evaluator:
    """Run a TinyAgent over a Benchmark and aggregate the results."""

    def __init__(self, create_agent: Callable):
        """Initialize with a function that creates a new agent instance."""
        self.create_agent = create_agent

    def run(self, benchmark: Benchmark) -> dict:
        """Run the agent on each example in the benchmark and score the results."""

        # Run each example and collect results
        results = []
        for example in benchmark.examples:
            agent = self.create_agent()
            prediction = agent.run(example["task"]) or ""
            passed = benchmark.scorer(prediction, example)
            results.append(
                {
                    "prediction": prediction,
                    "passed": passed,
                }
            )

        # Aggregate pass rate
        if results:
            pass_rate = sum(result["passed"] for result in results) / len(results)
        else:
            pass_rate = 0.0

        # Return detailed results and overall pass rate
        return {
            "name": benchmark.name,
            "pass_rate": pass_rate,
            "results": results,
        }


def create_agent():
    """Create a new instance of TinyAgent"""
    return TinyAgent(
        llm=llm,
        memory=Memory(),
        tools=NativeTools(),
        planner=NativeReAct(),
    )

## 3 - Exact Match

In [13]:
import re

def exact_match_scorer(prediction: str, example: dict) -> bool:
    """Return True if the answer matches the prediction, False otherwise"""
    match = re.search(r"\b([A-J])\b", prediction.upper())
    return match.group(1) == example["expected"]

In [14]:
# Three examples from MMLU Pro
mmlu_pro = Benchmark(
    name="MMLU Pro",
    examples=[
        {
            "task": """Which of the following is the body cavity that contains the pituitary gland?
A) Ventral B) Dorsal C) Buccal D) Thoracic E) Pericardial F) Abdominal G) Spinal H) Pelvic I) Pleural J) Cranial
Answer with only the letter.""",
            "expected": "J",
        },
        {
            "task": """What is the approximate mean cranial capacity of Homo erectus?
A) 1200 cc B) under 650 cc C) 1700 cc D) 1350 cc E) just under 1000 cc F) 1500 cc G) under 500 cc H) about 800 cc I) just over 1100 cc J) about 900 cc
Answer with only the letter.""",
            "expected": "E",
        },
        {
            "task": """	
According to Moore’s “ideal utilitarianism,” the right action is the one that brings about the greatest amount of:
A) wealth. B) virtue. C) fairness. D) pleasure. E) peace. F) justice. G) happiness. H) power. I) good. J) knowledge.
Answer with only the letter.""",
            "expected": "I",
        },
    ],
    scorer=exact_match_scorer,
)

In [15]:
from rich import print

# Run evaluation
result = Evaluator(create_agent).run(mmlu_pro)
print(result)

{
    'name': 'MMLU Pro',
    'pass_rate': 0.3333333333333333,
    'results': [
        {'prediction': 'J', 'passed': True},
        {'prediction': 'H', 'passed': False},
        {'prediction': 'G', 'passed': False}
    ]
}

## 4 - Programmatic Check

In [8]:
def programmatic_scorer(prediction: str, example: dict) -> bool:
    """Check a prediction against its related check."""
    return example["check"](prediction)

In [9]:
# Three examples from IFeval
ifeval = Benchmark(
    name="IFeval",
    examples=[
        {
            "task": "Write me a funny song with less than 10 sentences for a proposal to build a new playground at my local elementary school.",
            "check": lambda text: sum(1 for c in text if c in ".!?") < 10,
        },
        {
            "task": "Write an ad copy for a new product, a digital photo frame that connects to your social media accounts and displays your photos. Respond with at most 150 words.",
            "check": lambda text: len(text.split()) <= 150,
        },
        {
            "task": "I am planning a trip to Japan, and I would like thee to write an itinerary for my journey in a Shakespearean style. You are not allowed to use any commas in your response.",
            "check": lambda text: "," not in text,
        },
    ],
    scorer=programmatic_scorer,
)

In [10]:
# Run evaluation
result = Evaluator(create_agent).run(ifeval)
print(result)

{
    'name': 'IFeval',
    'pass_rate': 1.0,
    'results': [
        {
            'prediction': '(Sung to the tune of "Twinkle Twinkle Little Star")\n\nOur playground equipment is 
*old*,\nA story of rust, brave and bold.\nThe swings are rusty, the slides are weak,\nSo much much fun we 
desperately seek!\n\nWe need a playground shiny and grand,\nThe coolest in all of the land!\nWith bouncy nets, 
tunnels galore,\nAnd slides that reach to the floor!\n\nHelp us build it, let hammers ring,\nA happy, jumpy, 
wonderful thing!',
            'passed': True
        },
        {
            'prediction': "📸 **Stop Stashing Memories. Start Seeing Them.** 🏡\n\nYour best moments should never 
fade, and they never should live only on your phone.\n\nIntroducing the [Product Name] — the smart digital frame 
that transforms your endless stream of digital memories into breathtaking living art. Forget manual uploads and 
dusty photo albums. Simply link your favorite social media accounts (Instagram, Facebook, etc.), and we do the 
rest.\n\nThe [Product Name] constantly curates, streams, and displays your family's journey, friend's adventures, 
and milestones—automatically. It’s like having a personal, always-updated gallery wall that never 
sleeps.\n\nExperience genuine connection, effortlessly displayed.\n\n**Get yours today and keep your memories 
always in view.**\n*Visit [Website/Store Name]*",
            'passed': True
        },
        {
            'prediction': "Hark gentle traveler of modern days\nA tapestry of wonders shall amaze\nFor thou dost 
seek the Isles of Nippon's grace\nA journey steeped in beauty time and space.\n\nThy days we craft with flourish 
and with rhyme\nA tour through ages glorious sublime.\n\n**The First Three Days Near Edo's Grand Port**\n\nOn 
dawn's bright breath thy pilgrimage shall start\nThe bustling heart where neon arts impart.\nThou shalt behold the 
markets rich and bold\nWhere tales of ancient commerce are enrolled.\nA ramen broth a symphony to taste\nNo subtle 
flavour shall be left unplaced.\nThen wanders thee through Ginza's fine array\nWhere merchants boast their fortunes
every day.\nObserve the fashion's fleeting graceful sway\nA marvel to the eye come break of day.\n\nFor supper's 
cheer thou dost peruse a bar\nWhere sake like liquid gold doth softly star.\nTo hear the chatter and the laughter 
rings\nThe joyous chorus that sweet memory brings.\nThy first taste drawn a revel grand and deep\nA promise that 
the ancient streets do keep.\n\n**By Day Four A Journey Westward Bright**\n\nNext journeying forth where Edo does 
retreat\nTo Haanhbo's shores where polished sands do greet.\nThy steps lead thee toward a tranquil sight\nA temple 
standing bathed in morning light.\nHere silence reigns a sacred gentle peace\nWhere worldly clamour finds a sweet 
release.\nThou shalt observe the garden's careful plot\nWhere painted stones and winding paths are wrought.\nA 
moment pause a breath a thoughtful sigh\nAs feathered minstrels overhead do fly.\n\n**Days Five Six And Seven Of 
Culture's Swirl**\n\nTo Kyoto's realm where ancient spirit rests\nA whispered magic through the wooded nests.\nThe 
Gion district doth tempt thy wanderings near\nTo see the geisha's phantom passing year.\nObserve the kimonos of 
colours bright and sheer\nA perfect drama banishing all fear.\nAmongst the temples' wooden grace so old\nThy heart 
shall feel a story yet untold.\nThe golden pavilion shines a lovely gleam\nA vibrant vision in a waking 
dream.\n\nOn paths of autumn leaf or springtime dew\nThou shalt walk softly in the shade of yew.\nBefriended Edo's 
spirit grand and vast\nThe journey's memory forever shall last.\n\nGo forth now friend embrace the wondrous 
sight\nAnd savor Japan's magic pure and bright.\nMay thy travels bring thee joy untold\nA tale most worthy for the 
years of old.",
            'passed': True
        }
    ]
}

## 5 - LLM-as-a-judge

In [9]:
judge = LLM(
    model="gemini-3.1-flash-lite", 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai", 
    api_key="MY_API_KEY"
)

In [6]:
def judge_scorer(prediction: str, example: dict) -> bool:
    """The LLM-as-a-judge scorer."""
    prompt = f"""
Score the response from 0.0 to 1.0.

Expected: {example["expected"]}
Response: {prediction}

Reply with only a single number.
"""
    response = judge.generate([{"role": "user", "content": prompt}])
    score = float(response.content.strip().split()[0])
    return score

In [7]:
# Three examples adapted from MMLU Pro
mmlu_pro = Benchmark(
    name="MMLU Pro",
    examples=[
        {
            "task": "Which body cavity contains the pituitary gland?",
            "expected": "the cranial cavity",
        },
        {
            "task": "What is the approximate mean cranial capacity of Homo erectus?",
            "expected": "just under 1000 cc",
        },
        {
            "task": "According to Moore's 'ideal utilitarianism,' the right action is the one that brings about the greatest amount of what?",
            "expected": "good",
        },
    ],
    scorer=judge_scorer,
)

In [8]:
# Run evaluation
result = Evaluator(create_agent).run(mmlu_pro)
print(result)

{'name': 'MMLU Pro', 'pass_rate': 0.9, 'results': [{'prediction': 'The pituitary gland is housed within the **sella turcica** (or pituitary fossa) of the **sphenoid bone**, which is located within the **caudal cranial cavity** (the skull).\n\nMore specifically, while it is technically within a bone structure *within* the head, in anatomical discussions about body cavities, it is located within the region supplied by the general head/cranial area.\n\n**In summary:**\n\n* **General Area:** Head/Cranial Cavity\n* **Specific Bone Site:** Sella turcica (on the sphenoid bone)', 'passed': 1.0}, {'prediction': 'The approximate mean cranial capacity of *Homo erectus* varied over time and geographically, but generally falls within the range of **600 to 1250 cubic centimeters ($\\text{cc}$)**.\n\nHowever, when providing a single "mean" estimate, many general scientific summaries tend to place the average capacity between **700 $\\text{cc}$ and 900 $\\text{cc}$**.\n\n***\n\n### Important Context:\

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

We added a single new module — `evaluator.py` — on top of the Chapter 6 agent. The agent itself did not change. That separation is the point: a clean boundary between the system under test and the system doing the testing lets you iterate on either side without breaking the other.

In [14]:
from illustrated_agents.chapters.ch7 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py                                                                                                    │
│ ├── evaluator.py  ← New (Run a `TinyAgent` over a `Benchmark` suite and score outcomes.)                        │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── planning.py                                                                                                 │
│ ├── toolbox.py                                                                                                  │
│ ├── tools.py                                                                                                    │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

Up next is Chapter 8 on Multi-Agent Collaboration.